# 01. Construcción del corpus integral de evidencia

Este notebook inicializa y audita el registro de artículos, informes, capítulos, proyectos, políticas, planes y normas sobre cambio climático y pesquerías. No descarga documentos automáticamente: primero establece trazabilidad, tipos de fuente y control de versiones.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import yaml


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'config' / 'taxonomy.yml').exists():
            return candidate
    raise FileNotFoundError('No se encontró config/taxonomy.yml')

ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT


In [ ]:
with (ROOT / 'config' / 'taxonomy.yml').open(encoding='utf-8') as stream:
    taxonomy = yaml.safe_load(stream)

print(taxonomy['project']['name'])
print('Tipos de fuente:', len(taxonomy['source_type']))
print('Corrientes de evidencia:', len(taxonomy['evidence_stream']))
print('Tipos de hallazgo:', len(taxonomy['finding_type']))


In [ ]:
DATA = ROOT / 'data'
for folder in ['raw', 'interim', 'processed', 'templates']:
    (DATA / folder).mkdir(parents=True, exist_ok=True)

template_paths = {
    'sources': DATA / 'templates' / 'source_registry.csv',
    'screening': DATA / 'templates' / 'screening_decisions.csv',
    'findings': DATA / 'templates' / 'evidence_findings.csv',
    'quality': DATA / 'templates' / 'quality_appraisal.csv',
}

tables = {name: pd.read_csv(path) for name, path in template_paths.items()}
pd.DataFrame({
    'table': list(tables),
    'rows': [len(df) for df in tables.values()],
    'columns': [len(df.columns) for df in tables.values()],
})


## Auditoría del registro de fuentes

Cada versión documental recibe un `source_id` único. Una ley revisada, una nueva versión de un informe o una traducción deben registrarse como entradas diferenciadas y enlazarse mediante `supersedes_source_id` cuando corresponda.

In [ ]:
sources = tables['sources'].copy()
required_columns = {
    'source_id', 'title', 'source_type', 'year', 'source_status',
    'primary_url', 'access_date', 'source_language'
}
missing_columns = sorted(required_columns - set(sources.columns))
if missing_columns:
    raise ValueError(f'Faltan columnas obligatorias: {missing_columns}')

if not sources.empty:
    duplicated = sources.loc[sources['source_id'].duplicated(keep=False), 'source_id'].tolist()
    if duplicated:
        raise ValueError(f'source_id duplicados: {duplicated}')

    invalid_types = sorted(set(sources['source_type'].dropna()) - set(taxonomy['source_type']))
    if invalid_types:
        raise ValueError(f'Tipos de fuente fuera de la taxonomía: {invalid_types}')

print(f'Registro válido: {len(sources)} fuentes')


## Regla de separación analítica

- `source_registry.csv` contiene una fila por fuente o versión.
- `screening_decisions.csv` conserva inclusión, exclusión y revisión.
- `evidence_findings.csv` contiene una fila por hallazgo verificable.
- `quality_appraisal.csv` aplica criterios distintos según el tipo de fuente.

Un artículo puede generar varios resultados; un proyecto puede generar actividades, productos y resultados; una norma puede generar varios mecanismos. No deben comprimirse en una única fila documental.

In [ ]:
from evidence_review.quality import appraisal_criteria, appraisal_domain
from evidence_review.schema import SourceType

examples = [
    SourceType.SCIENTIFIC_ARTICLE,
    SourceType.TECHNICAL_REPORT,
    SourceType.PROJECT_EVALUATION,
    SourceType.FISHERY_MANAGEMENT_PLAN,
    SourceType.NATIONAL_LAW,
]

pd.DataFrame([
    {
        'source_type': source_type.value,
        'appraisal_domain': appraisal_domain(source_type),
        'n_criteria': len(appraisal_criteria(source_type)),
    }
    for source_type in examples
])


## Siguiente paso

Poblar el registro semilla, documentar la estrategia de búsqueda y crear el flujo de deduplicación. La descarga, extracción de texto y clasificación deben ejecutarse después de verificar licencias, versiones y fuentes primarias.